# 03 — Feature Engineering

Build per-trader behavioral profiles.

In [ ]:
import pathlib, sys, subprocess
sys.path.insert(0, str(pathlib.Path(".").resolve()))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [ ]:
result = subprocess.run([sys.executable, "src/features/profile_builder.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
profiles = pd.read_parquet("data/processed/trader_profiles.parquet")
print(profiles.shape)
print(profiles.dtypes)
profiles.describe()

In [ ]:
# Feature correlation heatmap
FEAT_COLS = [c for c in profiles.columns if c not in ("taker","is_leaked_market","case_id")]
corr = profiles[FEAT_COLS].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("notebooks/fig_feature_corr.png", dpi=150)
plt.show()

In [ ]:
# Feature distributions: leaked vs control
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, feat in zip(axes.flat, FEAT_COLS):
    for label, mask, c in [("Leaked", profiles["is_leaked_market"]==1, "crimson"), ("Control", profiles["is_leaked_market"]==0, "steelblue")]:
        ax.hist(profiles.loc[mask, feat].clip(lower=profiles[feat].quantile(0.01), upper=profiles[feat].quantile(0.99)), bins=30, alpha=0.5, label=label, color=c, density=True)
    ax.set_title(feat, fontsize=8)
    ax.set_xlabel("")
axes.flat[0].legend(fontsize=7)
for ax in axes.flat[len(FEAT_COLS):]:
    ax.set_visible(False)
plt.suptitle("Feature distributions: leaked vs control traders")
plt.tight_layout()
plt.savefig("notebooks/fig_feature_dists.png", dpi=150)
plt.show()